# 03 — Full-panel ARIMA, SARIMA, and LightGBM

Fit and evaluate every model on all 30,490 series, using parallel CPU workers and resumable checkpoints.

## Imports

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, json, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'data/calendar.csv').exists())
DATA = ROOT / 'data'
OUT = DATA / 'processed'
RESULTS = ROOT / 'reports/results'
FIGURES = ROOT / 'reports/figures'
for folder in [OUT, RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.venv/matplotlib'))


'/Users/moha/Machine-Learning-Project/Project 1/.venv/matplotlib'

In [3]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':100, 'axes.spines.top':False, 'axes.spines.right':False})
COLORS = ['#2563eb', '#059669', '#ea580c', '#7c3aed']


## Load prepared arrays

In [4]:
sales = np.load(OUT / 'sales.npy', mmap_mode='r')
prices = np.load(OUT / 'prices.npy', mmap_mode='r')
metadata = pd.read_csv(OUT / 'metadata.csv')
calendar = pd.read_csv(OUT / 'calendar.csv', parse_dates=['date'])
config = json.loads((ROOT / 'configs/comparison.json').read_text())
data = SimpleNamespace(root=ROOT)


## ARIMA, SARIMA, and LightGBM

In [5]:
import joblib
import lightgbm as lgb
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
panel=pd.read_csv(OUT / 'comparison_series.csv')
panel_rows=panel.row.to_numpy()
assert len(panel)==len(metadata)==30490 and panel.series_id.tolist()==metadata.series_id.iloc[panel_rows].tolist()
display(pd.Series(config).to_frame('setting'))


,setting
horizon,28
development_cutoffs,"[1857, 1885]"
test_cutoff,1913
history_days,728
training_stride,13
seed,42
n_estimators,200
num_leaves,31
learning_rate,0.05
baseline_floor,0.05


## Calendar features

In [6]:
CATEGORICAL = ['item_id_code','store_id_code','dept_id_code','cat_id_code',
               'state_id_code','weekday','month','event_1','event_2']


In [7]:
def calendar_features(rows, days):
    cal = calendar.iloc[days-1]
    meta = metadata.iloc[rows]
    f = meta[CATEGORICAL[:5]].reset_index(drop=True).astype('int16')
    f['weekday'] = cal.wday.to_numpy(dtype='int8')
    f['month'] = cal.month.to_numpy(dtype='int8')
    f['event_1'] = cal.event_name_1_code.to_numpy(dtype='int16')
    f['event_2'] = cal.event_name_2_code.to_numpy(dtype='int16')
    f['trend'] = (days-1).astype('float32')
    angle = 2*np.pi*cal.date.dt.day_of_year.to_numpy()/365.25
    f['annual_sin'], f['annual_cos'] = np.sin(angle).astype('float32'), np.cos(angle).astype('float32')
    f['snap'] = np.select([meta.state_id.to_numpy()==s for s in ['CA','TX','WI']],
                          [cal['snap_'+s].to_numpy() for s in ['CA','TX','WI']]).astype('int8')
    return f


## Lag and rolling features

In [8]:
def demand_features(rows, days):
    t = days-1
    f = {f'lag_{lag}':sales[rows,t-lag].astype('float32') for lag in [28,35,42,56]}
    window = sales[rows[:,None],t[:,None]-28-np.arange(28)]
    f['mean_7'] = window[:,:7].mean(axis=1,dtype='float32')
    f['mean_28'] = window.mean(axis=1,dtype='float32')
    f['std_28'] = window.std(axis=1,dtype='float32')
    f['zero_rate_28'] = (window==0).mean(axis=1).astype('float32')
    return pd.DataFrame(f)


## Features available at the forecast origin

In [9]:
def origin_features(rows, days, origins):
    rows,days,origins = map(np.asarray,(rows,days,origins))
    assert ((days-origins>=1)&(days-origins<=28)).all() and origins.min()>=56
    f = pd.concat([calendar_features(rows,days),demand_features(rows,days)],axis=1)
    recent = sales[rows[:,None],origins[:,None]-1-np.arange(28)]
    last_weekday = origins-1-((origins-days)%7)
    weekly = sales[rows[:,None],last_weekday[:,None]-7*np.arange(8)]
    f['horizon'] = (days-origins).astype('int8')
    f['origin_mean_7'] = recent[:,:7].mean(axis=1,dtype='float32')
    f['origin_mean_28'] = recent.mean(axis=1,dtype='float32')
    f['origin_zero_rate_28'] = (recent==0).mean(axis=1).astype('float32')
    f['weekday_baseline'] = weekly.mean(axis=1,dtype='float32')
    return f


## Training dates and forecast indices

In [10]:
def training_indices(cutoff):
    rows,days = [],[]
    start = cutoff-config['history_days']+1
    for offset in range(config['training_stride']):
        group = np.arange(offset,len(metadata),config['training_stride'])
        targets = np.arange(start+offset,cutoff+1,config['training_stride'])
        rows.append(np.repeat(group,len(targets)))
        days.append(np.tile(targets,len(group)))
    return np.concatenate(rows),np.concatenate(days)


In [11]:
def forecast_indices(panel_rows, cutoff):
    rows = np.repeat(np.asarray(panel_rows),config['horizon'])
    days = np.tile(np.arange(cutoff+1,cutoff+config['horizon']+1),len(panel_rows))
    return rows,days,np.full(len(rows),cutoff)


## Comparable error metrics

In [12]:
def rmsse_scales(history):
    scales=[]
    for values in history:
        active=np.flatnonzero(values)
        differences=np.diff(values[active[0]:].astype('float64')) if len(active) else []
        scales.append(float(np.mean(np.square(differences))) if len(differences) else 0.)
    return np.asarray(scales)


In [13]:
def score_forecast(prediction, actual, scales):
    assert prediction.shape==actual.shape and np.isfinite(prediction).all()
    assert (prediction>=0).all()
    error=prediction.astype('float64')-actual
    valid=scales>0
    rmsse=np.sqrt(np.mean(error[valid]**2,axis=1)/scales[valid])
    return dict(mae=float(np.abs(error).mean()),rmse=float(np.sqrt(np.mean(error**2))),
                mean_rmsse=float(rmsse.mean()),rmsse_series=int(valid.sum()),
                all_series=len(actual))


## Check RMSSE against hand calculations

In [14]:
toy=np.array([[0,0,2,4,3],[1,2,3,4,5]],dtype='float32')
np.testing.assert_allclose(rmsse_scales(toy),[2.5,1.0])
actual=np.ones((2,28))
assert score_forecast(actual,actual,np.ones(2))['mean_rmsse']==0
assert score_forecast(actual+1,actual,np.ones(2))['mean_rmsse']==1


## LightGBM settings

In [15]:
model_parameters=dict(objective='poisson',random_state=config['seed'],verbosity=-1,
    n_estimators=config['n_estimators'],learning_rate=config['learning_rate'],
    num_leaves=config['num_leaves'],max_bin=63,min_child_samples=100,
    reg_lambda=1.0,colsample_bytree=0.9,n_jobs=1)
MODEL_DIR=ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)


## Train the pooled LightGBM correction

In [16]:
def fit_lightgbm(cutoff):
    started=time.perf_counter()
    rows,days=training_indices(cutoff)
    assert days.max()<=cutoff and len(np.unique(rows))==len(metadata)
    horizons=(rows*17+days*13)%28+1
    X=origin_features(rows,days,days-horizons)
    offsets=np.maximum(X.weekday_baseline.to_numpy(),config['baseline_floor'])
    y=sales[rows,days-1].astype('float32')/offsets
    fitted=lgb.LGBMRegressor(**model_parameters)
    fitted.fit(X,y,categorical_feature=CATEGORICAL,sample_weight=offsets)
    joblib.dump(fitted,MODEL_DIR / f'comparison_lightgbm_d{cutoff}.joblib')
    return fitted,time.perf_counter()-started


## Predict the full panel with LightGBM

In [17]:
def predict_lightgbm(fitted, cutoff):
    prediction=np.empty((len(panel),config['horizon']),dtype='float32')
    for start in range(0,len(panel),3000):
        rows,days,origins=forecast_indices(panel_rows[start:start+3000],cutoff)
        X=origin_features(rows,days,origins)
        offsets=np.maximum(X.weekday_baseline.to_numpy(),config['baseline_floor'])
        values=np.maximum(fitted.predict(X)*offsets,0)
        prediction[start:start+3000]=values.reshape(-1,config['horizon'])
    return prediction


## Statistical models and CPU worker controls

In [18]:
from joblib import Parallel, delayed, parallel_config
from threadpoolctl import threadpool_limits
import hashlib
PREDICTIONS=DATA / 'interim'
PREDICTIONS.mkdir(exist_ok=True)


In [19]:
def fit_optimizer(history,kind,maxiter,diffuse=False,solver='lbfgs'):
    if kind=='ARIMA':
        model=ARIMA(history,order=tuple(config['arima_order']),trend='n')
    else:
        model=SARIMAX(history,order=tuple(config['arima_order']),
                      seasonal_order=tuple(config['seasonal_order']),trend='n')
    if diffuse:
        model.ssm.initialize_approximate_diffuse()
    if kind=='ARIMA':
        return model.fit(method_kwargs={'maxiter':maxiter,'method':solver})
    return model.fit(maxiter=maxiter,disp=False,method=solver)


In [20]:
def valid_fitted_forecast(fitted):
    values=np.asarray(fitted.forecast(config['horizon']),dtype='float32')
    return np.isfinite(values).all()


In [21]:
def fit_with_initialization_retry(history,kind):
    attempts=[(False,'lbfgs','fitted'),(True,'lbfgs','diffuse_retry'),
              (True,'powell','optimizer_retry')]
    for diffuse,solver,status in attempts:
        try:
            fitted=fit_optimizer(history,kind,config['maxiter'],diffuse,solver)
            if not fitted.mle_retvals.get('converged',True):
                fitted=fit_optimizer(history,kind,config['maxiter']*2,diffuse,solver)
            if not valid_fitted_forecast(fitted):
                raise FloatingPointError('Non-finite model or forecast')
            return fitted,status
        except (np.linalg.LinAlgError,ValueError,FloatingPointError) as error:
            last_error=error
    raise RuntimeError('All numerical retries failed') from last_error


## Fit a series and record optimizer diagnostics

In [22]:
def fit_statistical(history,kind):
    if np.ptp(history)==0:
        return np.full(config['horizon'],history[-1]),dict(
            status='constant',converged=False,warnings=0,negative_raw_forecasts=0)
    with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=1):
        warnings.simplefilter('always')
        fitted,status=fit_with_initialization_retry(history,kind)
    raw=np.asarray(fitted.forecast(config['horizon']))
    assert np.isfinite(raw).all()
    info=dict(status=status,converged=bool(fitted.mle_retvals.get('converged',True)),
              warnings=len(captured),negative_raw_forecasts=int((raw<0).sum()))
    return np.maximum(raw,0),info


## Identify the exact configuration and input data

In [23]:
import ast, importlib.metadata
input_info={p.name:dict(bytes=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns)
            for p in sorted(DATA.glob('*.csv'))}
source=json.loads((ROOT / 'notebooks/modeling/03_model_comparison.ipynb').read_text())
code_text='\n'.join(''.join(x['source']) for x in source['cells'] if x['cell_type']=='code')
critical=['fit_optimizer','fit_with_initialization_retry','valid_fitted_forecast',
          'fit_statistical','fit_row','fit_batch']
estimator_code='|'.join(ast.dump(node,include_attributes=False)
    for node in ast.parse(code_text).body if isinstance(node,ast.FunctionDef) and node.name in critical)
identity=dict(config=config,inputs=input_info,
    code_hash=hashlib.sha256(code_text.encode()).hexdigest(),
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','statsmodels','lightgbm']})


In [24]:
cache_config={key:config[key] for key in ['arima_order','seasonal_order',
    'history_days','maxiter','horizon','batch_size','run_version']}
cache_identity=dict(config=cache_config,inputs=input_info,versions=identity['versions'],
                    estimator_hash=hashlib.sha256(estimator_code.encode()).hexdigest())
run_id=hashlib.sha256(json.dumps(cache_identity,sort_keys=True).encode()).hexdigest()[:16]
CHECKPOINTS=PREDICTIONS / ('full_panel_checkpoints_'+run_id)
CHECKPOINTS.mkdir(exist_ok=True)
print('Checkpoint run:',run_id)


Checkpoint run: a7d982227033ecec


## Write completed batches atomically

In [25]:
def checkpoint_name(kind,cutoff,start,stop):
    return CHECKPOINTS / f'{kind}_d{cutoff}_{start:05d}_{stop:05d}.npz'

def save_batch(path,forecast,details):
    frame=pd.DataFrame(details)
    arrays={column:frame[column].to_numpy() for column in frame}
    for column in frame:
        if frame[column].dtype.kind in 'OUS':
            arrays[column]=frame[column].to_numpy(dtype=str)
    temporary=path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary,forecast=forecast,**arrays)
    os.replace(temporary,path)


## Process one batch of product/store series

In [26]:
def fit_row(history,kind,row,cutoff):
    try:
        return fit_statistical(history,kind)
    except Exception as error:
        raise RuntimeError(f'{kind}, d_{cutoff}, series row {row}: {error}') from error


In [27]:
def fit_batch(kind,cutoff,start,stop):
    path=checkpoint_name(kind,cutoff,start,stop)
    if path.exists():
        return str(path)
    local_sales=np.load(OUT / 'sales.npy',mmap_mode='r')
    forecast=np.empty((stop-start,config['horizon']),dtype='float32')
    details=[]
    for position,row in enumerate(range(start,stop)):
        history=local_sales[row,cutoff-config['history_days']:cutoff].astype('float64')
        values,info=fit_row(history,kind,row,cutoff)
        forecast[position]=values
        details.append(dict(row=row,model=kind,cutoff=cutoff,**info))
    save_batch(path,forecast,details)
    return str(path)


## Restore forecasts in their original row order

In [28]:
def collect_batches(kind,cutoff,ranges):
    forecast=np.empty((len(panel),config['horizon']),dtype='float32')
    records=[]
    for start,stop in ranges:
        with np.load(checkpoint_name(kind,cutoff,start,stop),allow_pickle=False) as batch:
            assert batch['row'].tolist()==list(range(start,stop))
            values=batch['forecast']
            assert values.shape==(stop-start,config['horizon']) and np.isfinite(values).all()
            forecast[start:stop]=values
            records.append(pd.DataFrame({k:batch[k] for k in batch.files if k!='forecast'}))
    return forecast,pd.concat(records,ignore_index=True).to_dict('records')


## Run all series with parallel CPU workers

In [29]:
def predict_statistical(cutoff,kind):
    ranges=[(start,min(start+config['batch_size'],len(panel)))
            for start in range(0,len(panel),config['batch_size'])]
    pending=[(start,stop) for start,stop in ranges
             if not checkpoint_name(kind,cutoff,start,stop).exists()]
    done=len(ranges)-len(pending)
    print(f'{kind} d_{cutoff}: {done}/{len(ranges)} batches cached',flush=True)
    with parallel_config(backend='loky',inner_max_num_threads=1):
        stream=Parallel(n_jobs=config['workers'],return_as='generator_unordered')(
            delayed(fit_batch)(kind,cutoff,start,stop) for start,stop in pending)
        for _ in stream:
            done+=1
            print(f'{kind} d_{cutoff}: {done}/{len(ranges)} batches',flush=True)
    return collect_batches(kind,cutoff,ranges)


## Save comparable scores and predictions

In [30]:
records,fit_diagnostics=[],[]
cutoffs=[*config['development_cutoffs'],config['test_cutoff']]
PREDICTIONS=DATA / 'interim'
PREDICTIONS.mkdir(exist_ok=True)

def record_result(name, cutoff, prediction, seconds, cached_batches=0):
    actual=sales[panel_rows,cutoff:cutoff+config['horizon']]
    scales=rmsse_scales(sales[panel_rows,:cutoff])
    phase='local_test' if cutoff==config['test_cutoff'] else 'development'
    records.append(dict(model=name,cutoff=cutoff,phase=phase,wall_seconds_this_run=seconds,cached_batches=cached_batches,
                        **score_forecast(prediction,actual,scales)))
    np.save(PREDICTIONS / f'full_panel_{name}_d{cutoff}.npy',prediction)
    pd.DataFrame(records).to_csv(RESULTS / 'full_panel_scores.csv',index=False)


## Run the two development windows and the previous local test

In [31]:
for cutoff in cutoffs:
    fitted,elapsed=fit_lightgbm(cutoff)
    record_result('LightGBM',cutoff,predict_lightgbm(fitted,cutoff),elapsed)
    for name in ['ARIMA','SARIMA']:
        cached=sum(checkpoint_name(name,cutoff,start,min(start+config['batch_size'],len(panel))).exists()
                   for start in range(0,len(panel),config['batch_size']))
        started=time.perf_counter()
        prediction,diagnostics=predict_statistical(cutoff,name)
        fit_diagnostics.extend(diagnostics)
        record_result(name,cutoff,prediction,time.perf_counter()-started,cached_batches=cached)
        pd.DataFrame(fit_diagnostics).to_csv(PREDICTIONS / 'full_panel_fit_diagnostics.csv',index=False)
    print(pd.DataFrame(records).tail(3).to_string(index=False),flush=True)


ARIMA d_1857: 120/120 batches cached


SARIMA d_1857: 120/120 batches cached


   model  cutoff       phase  wall_seconds_this_run  cached_batches      mae     rmse  mean_rmsse  rmsse_series  all_series
LightGBM    1857 development              36.610515               0 1.038714 2.252882    0.737348         30490       30490
   ARIMA    1857 development               0.543200             120 1.040654 2.309305    0.733311         30490       30490
  SARIMA    1857 development               0.305596             120 1.037606 2.298532    0.732940         30490       30490


ARIMA d_1885: 107/120 batches cached


ARIMA d_1885: 108/120 batches


...

SARIMA d_1913: 117/120 batches


SARIMA d_1913: 118/120 batches


SARIMA d_1913: 119/120 batches


SARIMA d_1913: 120/120 batches


   model  cutoff      phase  wall_seconds_this_run  cached_batches      mae     rmse  mean_rmsse  rmsse_series  all_series
LightGBM    1913 local_test              41.297212               0 1.047335 2.153174    0.759525         30490       30490
   ARIMA    1913 local_test             561.726518               0 1.056983 2.218067    0.761730         30490       30490
  SARIMA    1913 local_test            1188.120165               0 1.053837 2.203009    0.761337         30490       30490


## Check full-panel identity and prediction integrity

In [32]:
scores=pd.DataFrame(records)
assert len(scores)==9 and set(scores.model)=={'ARIMA','SARIMA','LightGBM'}
for cutoff in cutoffs:
    for name in ['ARIMA','SARIMA','LightGBM']:
        values=np.load(PREDICTIONS / f'full_panel_{name}_d{cutoff}.npy')
        assert values.shape==(len(metadata),28) and np.isfinite(values).all() and (values>=0).all()
display(scores)


,model,cutoff,phase,wall_seconds_this_run,cached_batches,mae,rmse,mean_rmsse,rmsse_series,all_series
0,LightGBM,1857,development,36.610515,0,1.038714,2.252882,0.737348,30490,30490
1,ARIMA,1857,development,0.543200,120,1.040654,2.309305,0.733311,30490,30490
2,SARIMA,1857,development,0.305596,120,1.037606,2.298532,0.732940,30490,30490
3,LightGBM,1885,development,38.267795,0,1.025961,2.106944,0.738100,30490,30490
4,ARIMA,1885,development,73.583857,107,1.046050,2.267792,0.738679,30490,30490
5,SARIMA,1885,development,1212.686070,0,1.042466,2.245494,0.738354,30490,30490
6,LightGBM,1913,local_test,41.297212,0,1.047335,2.153174,0.759525,30490,30490
7,ARIMA,1913,local_test,561.726518,0,1.056983,2.218067,0.761730,30490,30490
8,SARIMA,1913,local_test,1188.120165,0,1.053837,2.203009,0.761337,30490,30490


## Development ranking

In [33]:
ranking=scores[scores.phase=='development'].groupby('model').agg(
    mean_rmsse=('mean_rmsse','mean'),mae=('mae','mean'),rmse=('rmse','mean'))
ranking=ranking.sort_values('mean_rmsse')
display(ranking)
print('Development ranking only; the previously inspected local test is a retrospective check.')


,mean_rmsse,mae,rmse
model,,,
SARIMA,0.735647,1.040036,2.272013
ARIMA,0.735995,1.043352,2.288549
LightGBM,0.737724,1.032338,2.179913


Development ranking only; the previously inspected local test is a retrospective check.


## Record settings and package versions

In [34]:
import platform
manifest=dict(identity,run_id=run_id,series_count=len(panel),python=platform.python_version(),
    series_ids_hash=hashlib.sha256('|'.join(panel.series_id).encode()).hexdigest())
provenance=CHECKPOINTS / 'compatible_cache.json'
manifest['checkpoint_provenance']=json.loads(provenance.read_text()) if provenance.exists() else {}
(RESULTS / 'full_panel_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')


7046